# EXP-2026-008 C4 follow-up on LUMIERE: where the scaffold gains or loses (CPU)

The region split of card `experiments/registry/EXP-2026-008.md` (C4 follow-up), on the same 25 accepted training-role pairs of EXP-2026-005 run r2 as run `c4_lumiere_v1`. The scaffold equals the copy outside its edit region, so each pair's score difference (scaffold minus copy, primary MAE) splits exactly into three regions: the pre-op core's ID 1 (DeepBraTumIA's "necrotic or non-enhancing"), its ID 3 (enhancing) and the 2-mm band. Every step calls the tested module `rhuh_synthesis/lumiere_scaffold_check.py` (`run_region_check`).

- **Inputs:** run r2's records and training-role exports, and run `c4_lumiere_v1`'s record, all read through the Drive API by ID (each file checked by MD5). Those folders are never written. The development exports are not downloaded.
- **Checks:** the cohort is checked as in C4, and each pair's split must reproduce its C4 score difference; anything else stops the run.
- **Scores** use the real post-op scan, so they are target-informed. They describe; they decide nothing. No post-op label is read.
- **Output:** one write-once record, uploaded through the Drive API to `MyDrive/LUMIERE_derived/EXP-2026-008/<RUN_TAG>` and confirmed by the server's MD5. The record names the pairs' patients: keep that folder private. The printed summary and table hold no patient name.

A CPU runtime is enough. On the laptop one pair took about 7 s, so expect roughly 5 to 10 minutes, the download included. Colab asks once for permission to use the Drive API. Keep the tab open until the last cell finishes.

**Clear all outputs before saving, sharing or committing this notebook.**

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-c4-regions"
RUN_TAG = "c4_lumiere_regions_v1"  # a new tag for every attempt; a run folder that holds files is refused
SOURCE_RUN = ("LUMIERE_derived", "EXP-2026-005", "r2")  # read only
C4_RUN = ("LUMIERE_derived", "EXP-2026-008", "c4_lumiere_v1")  # read only
OUT_RUN = ("LUMIERE_derived", "EXP-2026-008", RUN_TAG)  # created under MyDrive through the Drive API
EXPECTED_PAIRS = 25  # run r2 accepted all 25 of its training-role pairs
LOCAL_SOURCE = "/content/c4_source_r2"  # downloaded copies of run r2's files
LOCAL_C4 = "/content/c4_lumiere_v1_record"  # downloaded copy of the C4 record
LOCAL_RUN = f"/content/{RUN_TAG}"  # the originals of this run's outputs; uploaded to Drive at the end

## 1. Code from the fork, packages, Drive API, run folders

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

assert not Path(LOCAL_RUN).exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

from rhuh_synthesis.colab_packages import DATA  # the one package list; it imports nothing

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DATA], check=True)
commit = subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)
source = DriveRun.open(api, (), SOURCE_RUN, LOCAL_SOURCE, create=False)  # run r2, by ID; never written
c4_run = DriveRun.open(api, (), C4_RUN, LOCAL_C4, create=False)  # run c4_lumiere_v1, by ID; never written
run_drive = DriveRun.open(api, (), OUT_RUN, LOCAL_RUN, create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before the long part
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)

## 2. Run r2's records and training-role exports, and the C4 record, by ID

In [ ]:
for name in ("run.json", "split.json", "registration_qc.csv", "summary.json"):
    source.fetch(f"outputs/{name}")  # by ID, checked by MD5
outputs = Path(LOCAL_SOURCE) / "outputs"
r2 = json.loads((outputs / "summary.json").read_text(encoding="utf-8"))
assert r2["stopped_at"] is None and r2["exports"]["train"]["accepted"] == EXPECTED_PAIRS, "Run r2's summary differs from the card: stop and report"
say(f"run r2 training role: {r2['exports']['train']}")
files = source.fetch_folder("outputs/export/train")  # the training-role exports and their sidecars only
say(f"downloaded {files} files of the training-role exports")
c4_record = json.loads(c4_run.fetch("c4_lumiere_v1.json").read_text(encoding="utf-8"))  # by ID, checked by MD5
say(f"C4 record: {c4_record['summary']['patients']} pairs, source {c4_record['summary']['source']}")

## 3. The region split of every accepted training-role pair, and the write-once record

In [ ]:
from rhuh_synthesis.lumiere_scaffold_check import run_region_check

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
summary, table, path, digest = run_region_check(
    outputs, config, c4_record, Path(LOCAL_RUN) / f"{RUN_TAG}.json", source_run="EXP-2026-005 run r2",
    expected=EXPECTED_PAIRS, repo_root=WORK, progress=say,
)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")

## 4. Confirm the upload and print the summary and the table

In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_RUN))
print("SUMMARY")
print(json.dumps(summary, indent=1))
print("TABLE")
print(table)

## What to send back

Copy everything printed after `SUMMARY`, the `TABLE` included. They hold scores only, no patient name. If a cell stops with an error, send the error message. An error about the split, the export folder, a sidecar, a SHA-256 or the C4 record means the files did not match their records, and an error that a split does not reproduce its C4 difference means the two codes disagree: in both cases the run stopped on purpose.